# Week 4 — Robustness and transfer between ions
**Two questions:**

**A. Are the improvements real?** Week 2 → 2b → 3 were each measured on *one* train/test split. Here we repeat every model on **10 different framework-based splits** and report the mean ± standard deviation, and how often each model beats the previous one.

**B. Can the model predict an ion it has never seen?** This is the central question for beyond-lithium batteries, where data is scarce.
- **Zero-shot, known hosts:** train on all ions *except* the target ion, then predict the target ion. Its host frameworks may still appear in training with other ions.
- **Zero-shot, new hosts (strict):** additionally remove every target-ion host framework from training — a brand-new ion in brand-new materials.
- **Few-shot:** add a small share (5–50 %) of the target ion's frameworks back into training — *how many examples are enough?*

Uses the files saved in Weeks 2–3 (`features_week2b.csv`, `chgnet_fingerprints.csv`, `xgb_params.json`). No API key needed. A GPU runtime makes it faster but is not required.

In [ ]:
!pip -q install xgboost
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/electrode_voltage_ml"

In [ ]:
import json, warnings, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")

ION_ORDER = ["Li", "Na", "K", "Mg", "Ca", "Zn"]
PALETTE = dict(zip(ION_ORDER, sns.color_palette("tab10", len(ION_ORDER))))

feat = pd.read_csv(f"{DATA_DIR}/features_week2b.csv")
FP = pd.read_csv(f"{DATA_DIR}/chgnet_fingerprints.csv")
params = json.load(open(f"{DATA_DIR}/xgb_params.json"))["best_params"]
assert len(feat) == len(FP)

y = feat["average_voltage"].values
ions = feat["working_ion"].values
groups = feat["framework"].values
X_2b = feat.drop(columns=["average_voltage", "working_ion", "framework"])
ox_cols = [c for c in X_2b.columns if c.startswith("ox_") or c in ("no_redox_metal", "elemental_host", "anion_free_host")]
ion_cols = [c for c in X_2b.columns if c.startswith("ion_")] + ["frac_ion_discharge"]
FEATURE_SETS = {
    "Week 2 (composition)": X_2b.drop(columns=ox_cols),
    "Week 2b (+ oxidation)": X_2b,
    "CHGNet structure + ion": pd.concat([FP, X_2b[ion_cols]], axis=1),
    "Combined (2b + CHGNet)": pd.concat([X_2b, FP], axis=1),
}

try:
    import torch; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"
def new_model():
    return XGBRegressor(**params, min_child_weight=2, reg_lambda=1.0, random_state=0, n_jobs=-1,
                        tree_method="hist", device=DEVICE)
print(f"{len(y)} electrodes | XGBoost on {DEVICE} | settings: {params}")

## A. Robustness: 10 repeated framework-based splits
Each repeat uses a different random seed. The "Mean per ion" reference is included so every number has context. (~5–15 minutes on CPU, faster on GPU.)

In [ ]:
N_REPEATS = 10
rows = []
t0 = time.time()
for seed in range(N_REPEATS):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(X_2b, y, groups))
    ion_mean = pd.Series(y[tr]).groupby(ions[tr]).mean()
    p = pd.Series(ions[te]).map(ion_mean).fillna(y[tr].mean()).values
    rows.append({"seed": seed, "model": "Mean per ion", "MAE": mean_absolute_error(y[te], p), "R2": r2_score(y[te], p)})
    for name, X in FEATURE_SETS.items():
        m = new_model().fit(X.iloc[tr], y[tr]); p = m.predict(X.iloc[te])
        rows.append({"seed": seed, "model": name, "MAE": mean_absolute_error(y[te], p), "R2": r2_score(y[te], p)})
    print(f"repeat {seed + 1}/{N_REPEATS} done ({time.time() - t0:.0f} s)")
rep = pd.DataFrame(rows)
order = ["Mean per ion"] + list(FEATURE_SETS)
summary = rep.groupby("model")[["MAE", "R2"]].agg(["mean", "std"]).reindex(order).round(3)
summary

In [ ]:
wide = rep.pivot(index="seed", columns="model", values="MAE")
pairs = [("Week 2 (composition)", "Week 2b (+ oxidation)"),
         ("Week 2b (+ oxidation)", "Combined (2b + CHGNet)"),
         ("Week 2b (+ oxidation)", "CHGNet structure + ion")]
print(f"Paired comparison over the same {N_REPEATS} splits:")
for a, b in pairs:
    d = wide[b] - wide[a]
    print(f"  {b:<26} vs {a:<24}: mean change {d.mean():+.3f} V (± {d.std():.3f}), better in {int((d < 0).sum())}/{len(d)} splits")

fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=rep, x="model", y="MAE", order=order, color="#8DA0CB", ax=ax)
sns.stripplot(data=rep, x="model", y="MAE", order=order, color="k", size=3, ax=ax)
ax.set(xlabel="", ylabel="Test MAE (V)", title=f"Test error over {N_REPEATS} framework-based splits")
plt.xticks(rotation=15, ha="right"); plt.tight_layout(); plt.savefig("fig_repeated_splits.png", dpi=200); plt.show()

## B1. Zero-shot transfer: predicting an ion the model has never seen
For each target ion we train on **all other ions** and predict **every** electrode of the target ion.
- **Known hosts:** target-ion frameworks may appear in training with other ions
- **New hosts (strict):** those frameworks are removed from training too

We report MAE, R², and the **bias** (mean predicted − DFT voltage): a positive bias means the model systematically over-estimates that ion.

*Note:* tree models cannot extrapolate beyond feature values seen in training. K has the largest ionic radius and Zn/Mg/Ca are the only divalent ions, so leaving them out tests exactly this limit.

In [ ]:
ZS_SETS = ["Week 2b (+ oxidation)", "Combined (2b + CHGNet)"]
# in-distribution reference: per-ion error from the repeated splits (Combined model)
ref_rows = []
for seed in range(N_REPEATS):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(X_2b, y, groups))
    p = new_model().fit(FEATURE_SETS["Combined (2b + CHGNet)"].iloc[tr], y[tr]).predict(FEATURE_SETS["Combined (2b + CHGNet)"].iloc[te])
    for i in ION_ORDER:
        m = ions[te] == i
        if m.sum() > 3: ref_rows.append({"ion": i, "MAE": mean_absolute_error(y[te][m], p[m])})
in_dist = pd.DataFrame(ref_rows).groupby("ion")["MAE"].mean()

zs_rows, zs_preds = [], {}
for target in ION_ORDER:
    test = ions == target
    target_fw = set(groups[test])
    for setting in ["known hosts", "new hosts (strict)"]:
        train = ~test if setting == "known hosts" else (~test) & ~np.isin(groups, list(target_fw))
        if train.sum() < 100:
            print(f"  skipping {target} / {setting}: only {int(train.sum())} training electrodes left")
            continue
        for name in ZS_SETS:
            X = FEATURE_SETS[name]
            p = new_model().fit(X[train], y[train]).predict(X[test])
            zs_rows.append({"target ion": target, "setting": setting, "model": name, "n_train": int(train.sum()),
                            "n_test": int(test.sum()), "MAE": mean_absolute_error(y[test], p),
                            "R2": r2_score(y[test], p), "bias": float(np.mean(p - y[test]))})
            zs_preds[(target, setting, name)] = p
    print(f"{target} done")
zs = pd.DataFrame(zs_rows)
tab = zs[zs["model"] == "Combined (2b + CHGNet)"].pivot(index="target ion", columns="setting", values="MAE").reindex(ION_ORDER)
tab.insert(0, "in-distribution", in_dist.reindex(ION_ORDER))
print("Test MAE (V), Combined model:"); tab.round(3)

In [ ]:
print("Full zero-shot results (bias = mean predicted - DFT):")
zs.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
tab.plot.bar(ax=ax, color=["#66C2A5", "#FC8D62", "#8DA0CB"], width=0.8)
ax.set(xlabel="Target ion", ylabel="Test MAE (V)", title="How well does the model transfer to an unseen ion?")
ax.legend(title=""); plt.xticks(rotation=0); plt.tight_layout(); plt.savefig("fig_zero_shot.png", dpi=200); plt.show()

fig, axs = plt.subplots(1, 3, figsize=(15, 4.8), sharex=True, sharey=True)
for a, t in zip(axs, ["Na", "K", "Mg"]):
    m = ions == t
    p = zs_preds[(t, "known hosts", "Combined (2b + CHGNet)")]
    a.scatter(y[m], p, s=12, alpha=0.7, color=PALETTE[t])
    a.plot([-0.2, 6.2], [-0.2, 6.2], "k--", lw=1)
    r = zs[(zs["target ion"] == t) & (zs["setting"] == "known hosts") & (zs["model"] == "Combined (2b + CHGNet)")].iloc[0]
    a.set(xlim=(-0.2, 6.2), ylim=(-0.2, 6.2), xlabel="DFT average voltage (V)",
          title=f"{t}, never seen in training\nMAE {r.MAE:.2f} V, bias {r.bias:+.2f} V")
axs[0].set_ylabel("Predicted voltage (V)")
plt.tight_layout(); plt.savefig("fig_zero_shot_parity.png", dpi=200); plt.show()

## B2. Few-shot: how many examples of the new ion are enough?
For Na, K and Mg we add back a small share of the target ion's **frameworks** (5, 10, 25, 50 %) into training and test on the remaining target-ion electrodes. Repeated 5 times with different random choices.

In [ ]:
FRACTIONS = [0.0, 0.05, 0.10, 0.25, 0.50]
N_FS = 5
X = FEATURE_SETS["Combined (2b + CHGNet)"]
fs_rows = []
for target in ["Na", "K", "Mg"]:
    tgt = np.where(ions == target)[0]
    fw = np.array(sorted(set(groups[tgt])))
    for frac in FRACTIONS:
        for rep_i in range(N_FS if frac > 0 else 1):
            rng = np.random.default_rng(rep_i)
            chosen = set(rng.choice(fw, size=int(round(frac * len(fw))), replace=False)) if frac > 0 else set()
            add = np.array([i for i in tgt if groups[i] in chosen], dtype=int)
            held = np.array([i for i in tgt if groups[i] not in chosen], dtype=int)
            train = np.concatenate([np.where(ions != target)[0], add])
            p = new_model().fit(X.iloc[train], y[train]).predict(X.iloc[held])
            fs_rows.append({"target ion": target, "fraction": frac, "n_added": len(add),
                            "MAE": mean_absolute_error(y[held], p)})
    print(f"{target} done")
fs = pd.DataFrame(fs_rows)
fs_sum = fs.groupby(["target ion", "fraction"]).agg(MAE=("MAE", "mean"), MAE_std=("MAE", "std"),
                                                    n_added=("n_added", "mean")).round(3)
fs_sum

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for t in ["Na", "K", "Mg"]:
    d = fs_sum.loc[t].reset_index()
    ax.errorbar(d["fraction"] * 100, d["MAE"], yerr=d["MAE_std"].fillna(0), marker="o", capsize=3,
                label=t, color=PALETTE[t])
    ax.axhline(in_dist.get(t, np.nan), color=PALETTE[t], ls=":", lw=1)
ax.set(xlabel="Share of target-ion frameworks added to training (%)", ylabel="Test MAE on remaining electrodes (V)",
       title="Few-shot learning curve (dotted = normal in-distribution error)")
ax.legend(title="Target ion"); plt.tight_layout(); plt.savefig("fig_few_shot.png", dpi=200); plt.show()

## Save

In [ ]:
import shutil
rep.to_csv("week4_repeated_splits.csv", index=False); summary.to_csv("week4_repeated_summary.csv")
zs.to_csv("week4_zero_shot.csv", index=False); tab.to_csv("week4_zero_shot_table.csv")
fs.to_csv("week4_few_shot.csv", index=False); fs_sum.to_csv("week4_few_shot_summary.csv")
for f in ["week4_repeated_splits.csv", "week4_repeated_summary.csv", "week4_zero_shot.csv", "week4_zero_shot_table.csv",
          "week4_few_shot.csv", "week4_few_shot_summary.csv", "fig_repeated_splits.png", "fig_zero_shot.png",
          "fig_zero_shot_parity.png", "fig_few_shot.png"]:
    shutil.copy(f, DATA_DIR)
print("Saved to", DATA_DIR)

## What to send me
1. The **repeated-splits summary** and the **paired comparison** lines
2. The **zero-shot table** (in-distribution vs known hosts vs new hosts) and the full zero-shot results with the **bias** column
3. The **few-shot summary**
4. The plots, if you like

From these we will write the final results and the conclusions of the project.